# Observed SWE: SNOTEL and CDEC

Builds daily and peak SWE datasets from station observations, for comparison with the UCLA reanalysis. All SWE is in **mm**.

**SNOTEL:** combines the per-station CSVs (converted to mm if needed) and adds station metadata.
**CDEC:** downloads daily revised SWE (sensor 82) for each station, converts it to mm, adds station metadata, and computes daily SWE gain.

**Quality control (same for both networks)**, before computing peak SWE per water year:
- remove negative values,
- remove day-to-day jumps of 1000 mm or more,
- keep January–May only,
- remove station-years with fewer than 7 observations or none in April,
- CDEC only: remove known bad values (SPS and EBB above 2500 mm).

## Inputs
| File | Contents |
|------|----------|
| `Data/SNOTEL/df_<id>.csv` | Daily SNOTEL data per station |
| `Data/SNOTEL/NRCS_SNOTEL_Locations_noAlaska.csv` | SNOTEL names, elevation, location |
| `Data/CDEC/cdec_snow_stations.csv` | CDEC station list: IDs, names, elevation [ft], location, record start |

## Outputs
| File | Contents |
|------|----------|
| `Data/CDEC/Revised/<id>_swe.csv` | Raw CDEC downloads |
| `Data/Cleaned/SNOTEL/snotel_ds.nc4` | Daily `SWE` [mm] (Station_ID, Day) + metadata, before QC |
| `Data/Cleaned/SNOTEL/snotel_peak_swe.nc4` | Peak `SWE` [mm] (Station_ID, WY), after QC |
| `Data/Cleaned/CDEC/cdec_ds.nc4` | Daily `SWE` and `Delta_SWE` [mm] (Station_ID, Day) + metadata, before QC |
| `Data/Cleaned/CDEC/cdec_peak_swe.nc4` | Peak `SWE` [mm] (Station_ID, WY), after QC |

In [ ]:
# %% Imports
import glob
import os
import re
from io import StringIO


import pandas as pd
import requests
import xarray as xr

from config import data_folder
from utils import calculate_water_year

snotel_out = os.path.join(data_folder, 'Data/Cleaned/SNOTEL')
cdec_out = os.path.join(data_folder, 'Data/Cleaned/CDEC')
os.makedirs(snotel_out, exist_ok=True)
os.makedirs(cdec_out, exist_ok=True)

In [ ]:
# %% Quality control shared by both networks
def qc_swe(swe, max_jump=1000, min_obs=7):
    """Peak-SWE quality control for daily SWE [mm] with a WY coordinate on Day:
    drop day-to-day jumps >= max_jump, keep Jan–May, and drop station-years
    with fewer than min_obs observations or none in April."""
    swe = remove_swe_jumps(swe, max_jump)
    swe = swe.where(swe['Day'].dt.month <= 5)
    
    n_obs = swe.notnull().groupby('WY').sum()
    has_april = (swe.notnull() & (swe['Day'].dt.month == 4)).groupby('WY').any()
    return swe.where(((n_obs >= min_obs) & has_april).sel(WY=swe['WY']))

In [ ]:
# %% ===== SNOTEL =====
SWE_COLUMN = re.compile(r'Snow Water Equivalent.*Start of Day Values$')


def read_snotel(path):
    """One SNOTEL CSV (df_<id>.csv) -> SWE [mm] over Day for one Station_ID."""
    station_id = int(os.path.basename(path)[len('df_'):-len('.csv')])
    df = pd.read_csv(path, index_col=0)
    swe_col = next(c for c in df.columns if SWE_COLUMN.search(c))
    if '(in)' in swe_col:
        to_mm = 25.4
    elif '(mm)' in swe_col:
        to_mm = 1.0
    else:
        raise ValueError(f'Unknown SWE units in column {swe_col!r} ({path})')
    return (xr.Dataset({'SWE': ('Day', df[swe_col].values * to_mm)}, coords={'Day': pd.to_datetime(df.index)})
              .expand_dims(Station_ID=[station_id]))


snotel = xr.concat([read_snotel(f) for f in glob.glob(os.path.join(data_folder, 'Data/SNOTEL/df_*.csv'))],
                   dim='Station_ID', join='outer')
snotel['SWE'] = snotel['SWE'].where(snotel['SWE'] >= 0)  # negative = sensor error; zeros are real

stations = pd.read_csv(os.path.join(data_folder, 'Data/SNOTEL/NRCS_SNOTEL_Locations_noAlaska.csv'),
                       index_col=0).loc[snotel.Station_ID.values]
snotel['Station_Name'] = ('Station_ID', stations['Station Name'].values)
for var in ['Latitude', 'Longitude']:
    snotel[var] = ('Station_ID', stations[var].values)
    snotel['Elevation'] = ('Station_ID', stations['Elevation'].values * 0.3048)  # ft -> m
snotel = snotel.assign_coords(WY=calculate_water_year(snotel['Day']))

snotel.to_netcdf(os.path.join(snotel_out, 'snotel_ds.nc4'))
qc_swe(snotel['SWE']).groupby('WY').max('Day', skipna=True).to_netcdf(
    os.path.join(snotel_out, 'snotel_peak_swe.nc4'))

In [ ]:
# %% ===== CDEC: station list =====
stations_raw = pd.read_csv(os.path.join(data_folder, 'Data/CDEC/cdec_snow_stations.csv'))

cdec_stations = (stations_raw[['HYDROLOGIC AREA,\n RIVER NAME, AND\nSNOW COURSE NAME', 'Station ID',
                               'ELEV\nIN\nFEET', 'LATITUDE', 'LONGITUDE']]
                 .set_axis(['Snow_Course_Name', 'Station_ID', 'Elevation', 'Latitude', 'Longitude'], axis=1)
                 .set_index('Station_ID'))
cdec_stations['Elevation'] *= 0.3048  # ft -> m
cdec_stations['Elevation'].agg(['mean', 'min', 'max'])

In [ ]:
# %% ===== CDEC: download daily revised SWE (sensor 82) =====
CDEC_URL = 'https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet'
SENSORS = ['82']            # 82 = snow water content, revised [in]
DEFAULT_START = '1930-01-01'
END = '2025-12-31'
raw_dir = os.path.join(data_folder, 'Data/CDEC/Revised')
os.makedirs(raw_dir, exist_ok=True)


def record_start(value):
    """'RECORD BEGAN' entry -> 'YYYY-MM-DD'; falls back to DEFAULT_START if empty or unreadable."""
    s = str(value).strip()
    if re.fullmatch(r'\d{4}(\.0)?', s):  # bare year, possibly read as a float
        s = s[:4]
    d = pd.to_datetime(s, errors='coerce')
    return DEFAULT_START if pd.isna(d) else d.strftime('%Y-%m-%d')


def download_cdec(station_id, start, end, sensors=SENSORS, out_dir=raw_dir):
    """Download daily data for one station (all `sensors`) to <out_dir>/<station_id>_swe.csv.
    Returns True if any data came back."""
    frames = []
    for sensor in sensors:
        r = requests.get(CDEC_URL, timeout=120, params={
            'Stations': station_id, 'SensorNums': sensor, 'dur_code': 'D', 'Start': start, 'End': end})
        if r.status_code == 200 and r.text.strip():
            frames.append(pd.read_csv(StringIO(r.text), dtype={7: str}).assign(Sensor=sensor))
    if frames:
        pd.concat(frames).to_csv(os.path.join(out_dir, f'{station_id}_swe.csv'), index=False)
    return bool(frames)


# One-off: rename files from earlier runs so they aren't downloaded again
for f in glob.glob(os.path.join(raw_dir, '*_ele.csv')):
    os.rename(f, f.replace('_ele.csv', '_swe.csv'))

no_data = []
for _, row in stations_raw.iterrows():
    sid = row['Station ID']
    if os.path.exists(os.path.join(raw_dir, f'{sid}_swe.csv')):
        continue  # delete the file to re-download
    if not download_cdec(sid, record_start(row.get('RECORD BEGAN')), END):
        no_data.append(sid)
print(f'{len(no_data)} stations returned no data: {no_data}')

In [ ]:
# %% ===== CDEC: combine downloads =====
def read_cdec(path):
    """One downloaded CDEC CSV -> SWE [in] over (Station_ID, Day)."""
    df = (pd.read_csv(path, usecols=['STATION_ID', 'DATE TIME', 'SENSOR_NUMBER', 'VALUE'])
            .set_axis(['Station_ID', 'Day', 'Data_Type', 'Value'], axis=1))
    df['Day'] = pd.to_datetime(df['Day'], format='%Y%m%d %H%M')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')  # '---' etc. -> NaN
    return (df.set_index(['Station_ID', 'Day', 'Data_Type'])['Value'].to_xarray()
              .to_dataset(dim='Data_Type').rename({82: 'SWE'}))


cdec = xr.concat([read_cdec(f) for f in glob.glob(os.path.join(raw_dir, '*_swe.csv'))],
                 dim='Station_ID', join='outer')
cdec['SWE'] = cdec['SWE'] * 25.4                     # in -> mm
cdec['SWE'] = cdec['SWE'].where(cdec['SWE'] >= 0)   # negative = sensor error; zeros are real

meta = cdec_stations.loc[cdec.Station_ID.values]
for var in ['Snow_Course_Name', 'Elevation', 'Latitude', 'Longitude']:
    cdec[var] = ('Station_ID', meta[var].values)

delta = cdec['SWE'].diff('Day', label='upper')
cdec['Delta_SWE'] = delta.where((delta >= 0) | cdec['SWE'].isnull(), 0)  # SWE gains only

cdec.to_netcdf(os.path.join(cdec_out, 'cdec_ds.nc4'))

In [ ]:
# %% Check: daily pillows (~100+ obs per Jan–May) or monthly snow courses (a handful)?
obs_per_year = cdec['SWE'].notnull().groupby(calculate_water_year(cdec['Day'])).sum()
print(f"{cdec.sizes['Station_ID']} of {len(stations_raw)} listed stations returned data")
obs_per_year.where(obs_per_year > 0).median('WY').to_series().describe()

In [ ]:
# %% ===== CDEC: quality control and peak SWE =====
cdec = xr.open_dataset(os.path.join(cdec_out, 'cdec_ds.nc4'))
cdec = cdec.assign_coords(WY=calculate_water_year(cdec['Day']))

swe = qc_swe(cdec['SWE'])
swe = swe.where(~(swe['Station_ID'].isin(['SPS', 'EBB']) & (swe > 2500)))  # known faulty values

swe.groupby('WY').max('Day', skipna=True).to_netcdf(os.path.join(cdec_out, 'cdec_peak_swe.nc4'))# %% Imports
